# MODELIZACION

In [2]:
import pandas as pd
import numpy as np


Leemos tabla:

In [3]:
config = {
    "archivo_datos": "Estadisticas Jugadores 2025-2026.xlsx",
    "nombre_hoja": 0,
}

def cargar_datos(ruta, hoja=None):
    if ruta.endswith(".csv"):
        df = pd.read_csv(ruta)
    else:
        df = pd.read_excel(ruta, sheet_name=hoja)
    print(f"  Datos cargados: {df.shape[0]} filas x {df.shape[1]} columnas")
    return df


Data engenieering:

POSITION 2 y POSITION 3 la logica sera la siguiente: si no tiene nada en position2, se rellenara con el dato de MAIN POSITION, lo mismo con position 3, este se rellenara con poistion 2, simple, HEIGHT se rellenara con la mediana, FOOT se rellenara con Derecho, CONTRACT UNTIL se rellenara con 30/06/2026, TEAM JOINED se rellenara con 30/06/2025, PREVIOUS TEAM se rellanara con unknown

In [4]:
def rellenar_position_2(df):
    df = df.copy()
    n = df["POSITION 2"].isna().sum()
    df["POSITION 2"] = df["POSITION 2"].fillna(df["MAIN POSITION"])
    print(f"  POSITION 2 rellenados con MAIN POSITION: {n}")
    return df


def rellenar_position_3(df):
    df = df.copy()
    n = df["POSITION 3"].isna().sum()
    df["POSITION 3"] = df["POSITION 3"].fillna(df["POSITION 2"])
    print(f"  POSITION 3 rellenados con POSITION 2: {n}")
    return df


def rellenar_height_mediana(df):
    df = df.copy()
    mediana = df["HEIGHT"].median()
    n = df["HEIGHT"].isna().sum()
    df["HEIGHT"] = df["HEIGHT"].fillna(mediana)
    print(f"  HEIGHT rellenados con mediana ({mediana}): {n}")
    return df


def rellenar_foot(df):
    df = df.copy()
    n = df["FOOT"].isna().sum()
    df["FOOT"] = df["FOOT"].fillna("Derecho")
    print(f"  FOOT rellenados con 'Derecho': {n}")
    return df


def rellenar_contract_until(df):
    df = df.copy()
    n = df["CONTRACT UNTIL"].isna().sum()
    df["CONTRACT UNTIL"] = df["CONTRACT UNTIL"].fillna("30/06/2026")
    print(f"  CONTRACT UNTIL rellenados con '30/06/2026': {n}")
    return df


def rellenar_team_joined(df):
    df = df.copy()
    n = df["TEAM JOINED"].isna().sum()
    df["TEAM JOINED"] = df["TEAM JOINED"].fillna("30/06/2025")
    print(f"  TEAM JOINED rellenados con '30/06/2025': {n}")
    return df


def rellenar_previous_team(df):
    df = df.copy()
    n = df["PREVIOUS TEAM"].isna().sum()
    df["PREVIOUS TEAM"] = df["PREVIOUS TEAM"].fillna("unknown")
    print(f"  PREVIOUS TEAM rellenados con 'unknown': {n}")
    return df

Eliminamos registros de jugadores con menos de 500 minutos para evitar datos anomalos:

In [5]:
def filtrar_mins(df, minimo=500):
    df = df.copy()
    antes = df.shape[0]
    df = df[df["MINS"] >= minimo]
    print(f"  Filas eliminadas (MINS < {minimo}): {antes - df.shape[0]}")
    print(f"  Filas restantes: {df.shape[0]}")
    return df

Ahora creamos dos variables, MARKET_VALUE_DROP que es la caida de valor de mercado que ha tenido, es decir, el valor maximo y el actual. Tambien se añade la misma variable pero en porcentaje.

In [6]:
def calcular_diferencia_market_value(df):
    df = df.copy()

    df["MARKET VALUE"] = pd.to_numeric(df["MARKET VALUE"], errors="coerce")
    df["HIGHEST MARKET VALUE"] = pd.to_numeric(df["HIGHEST MARKET VALUE"], errors="coerce")

    # Diferencia absoluta en millones de euros
    df["MARKET_VALUE_DROP"] = (df["HIGHEST MARKET VALUE"] - df["MARKET VALUE"]).round(2)

    # Tendencia porcentual: negativa si el valor actual ha caído respecto al máximo
    df["MARKET_VALUE_TREND_PCT"] = np.where(
        df["HIGHEST MARKET VALUE"] > 0,
        ((df["MARKET VALUE"] - df["HIGHEST MARKET VALUE"]) / df["HIGHEST MARKET VALUE"]) * 100,
        np.nan,
    ).round(2)

    print(f"  MARKET_VALUE_DROP y MARKET_VALUE_TREND_PCT calculados")
    return df



Porcentaje de titularidades:

In [7]:
def calcular_start_pct(df):
    df = df.copy()

    df["STARTS"] = pd.to_numeric(df["STARTS"], errors="coerce")
    df["BENCH"] = pd.to_numeric(df["BENCH"], errors="coerce")

    total = df["STARTS"] + df["BENCH"]
    df["START_PCT"] = np.where(
        total > 0,
        (df["STARTS"] / total) * 100,
        np.nan,
    ).round(2)

    print(f"  START_PCT calculado")
    return df

MINUTOS DISPONIBLES JUGADOS:

In [8]:
def calcular_available_min_pct(df):
    df = df.copy()

    df["STARTS"] = pd.to_numeric(df["STARTS"], errors="coerce")
    df["BENCH"] = pd.to_numeric(df["BENCH"], errors="coerce")
    df["MINS"] = pd.to_numeric(df["MINS"], errors="coerce")

    min_disponibles = (df["STARTS"] + df["BENCH"]) * 90
    df["AVAILABLE_MIN_PCT"] = (np.where(
        min_disponibles > 0,
        df["MINS"] / min_disponibles,
        np.nan,
    ).round(2))*100

    print(f"  AVAILABLE_MIN_PCT calculado")
    return df

Creamos CONTRACT_OPPORTUNITY que sea un 1 o un 0 dependiendo de si queda un año para que se acabe el contrato:

In [9]:
def calcular_situacion_contractual(df, fecha_referencia="30/06/2026"):
    df = df.copy()

    ref = pd.to_datetime(fecha_referencia, format="%d/%m/%Y")
    contract = pd.to_datetime(df["CONTRACT UNTIL"], format="%d/%m/%Y", errors="coerce")

    # Meses hasta el fin de contrato
    df["MONTHS_TO_CONTRACT_END"] = (
        (contract.dt.year - ref.year) * 12 + (contract.dt.month - ref.month)
    )

    # Oportunidad: 1 si quedan 12 meses o menos, 0 si quedan más
    df["CONTRACT_OPPORTUNITY"] = np.where(
        df["MONTHS_TO_CONTRACT_END"].notna(),
        (df["MONTHS_TO_CONTRACT_END"] <= 12).astype(int),
        np.nan,
    )

    print(f"  MONTHS_TO_CONTRACT_END y CONTRACT_OPPORTUNITY calculados")
    return df

Creamos las variables goles, asistencias y goles+asistencias por 90 min:

In [10]:
def calcular_contribuciones_90(df):
    df = df.copy()

    df["GOALS"] = pd.to_numeric(df["GOALS"], errors="coerce")
    df["ASSIST"] = pd.to_numeric(df["ASSIST"], errors="coerce")
    df["MINS"] = pd.to_numeric(df["MINS"], errors="coerce")

    df["GOALS_90"] = np.where(df["MINS"] > 0, 90 * df["GOALS"] / df["MINS"], np.nan).round(2)
    df["ASSISTS_90"] = np.where(df["MINS"] > 0, 90 * df["ASSIST"] / df["MINS"], np.nan).round(2)
    df["GOAL_CONTRIBUTIONS_90"] = np.where(
        df["MINS"] > 0,
        90 * (df["GOALS"] + df["ASSIST"]) / df["MINS"],
        np.nan,
    ).round(2)

    print(f"  GOALS_90, ASSISTS_90 y GOAL_CONTRIBUTIONS_90 calculados")
    return df

Creamos goles/tiro

In [11]:
def calcular_goal_conversion(df):
    df = df.copy()

    df["GOALS"] = pd.to_numeric(df["GOALS"], errors="coerce")
    df["SHOTS"] = pd.to_numeric(df["SHOTS"], errors="coerce")

    df["GOAL_CONVERSION"] = np.where(
        df["SHOTS"] > 0,
        df["GOALS"] / df["SHOTS"],
        0,
    ).round(2)

    print(f"  GOAL_CONVERSION calculado")
    return df

Sumamos acciones defensivas:

In [12]:
def calcular_defensive_actions(df):
    df = df.copy()

    cols_def = ["TACKLE", "INTERCEPTION", "CLEARANCE", "BLOCKS", "DRIBBLE_DEF"]
    for c in cols_def:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["DEFENSIVE_ACTIONS"] = df[cols_def].sum(axis=1, skipna=True).round(2)

    print(f"  DEFENSIVE_ACTIONS calculado (suma de {', '.join(cols_def)})")
    return df

Sumamos acciones de creacion de juego:

In [13]:
def calcular_chance_creation(df):
    df = df.copy()

    cols_creacion = ["KEY PASSES", "CROSS"]
    for c in cols_creacion:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["CHANCE_CREATION"] = df[cols_creacion].sum(axis=1, skipna=True).round(2)

    print(f"  CHANCE_CREATION calculado (suma de {', '.join(cols_creacion)})")
    return df

Sumamos regates y faltas recibidas:

In [14]:
def calcular_dribbling_threat(df):
    df = df.copy()

    cols = ["DRIBBLE_OF", "TACKLE RECEIVED"]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["DRIBBLING_THREAT"] = df[cols].sum(axis=1, skipna=True).round(2)

    print(f"  DRIBBLING_THREAT calculado (suma de {', '.join(cols)})")
    return df

Excluimos porteros del recomendador:

In [15]:
def excluir_porteros(df):
    df = df.copy()
    antes = df.shape[0]
    df = df[df["MAIN POSITION"] != "Portero"]
    print(f"  Porteros eliminados: {antes - df.shape[0]}")
    print(f"  Jugadores de campo restantes: {df.shape[0]}")
    return df

Excluimos jugadores con valor de mercado 0

In [16]:
def excluir_market_value_cero(df):
    df = df.copy()
    df["MARKET VALUE"] = pd.to_numeric(df["MARKET VALUE"], errors="coerce")
    antes = df.shape[0]
    df = df[df["MARKET VALUE"] > 0]
    print(f"  Jugadores con valor 0 o nulo eliminados: {antes - df.shape[0]}")
    print(f"  Jugadores restantes: {df.shape[0]}")
    return df

Eliminamos la variable nacionalidad y previus team:

In [17]:
def eliminar_columnas(df, columnas):
    df = df.copy()
    existentes = [c for c in columnas if c in df.columns]
    df = df.drop(columns=existentes)
    print(f"  Columnas eliminadas: {', '.join(existentes)}")
    return df

### BINARIZACION Y TRAMIFICACION:

Primero asignamos pesos a las posiciones para quitarnos el texto:

In [18]:
def codificar_posiciones_ponderadas(df, peso_main=1.0, peso_pos2=0.6, peso_pos3=0.3):
    df = df.copy()

    posiciones = pd.unique(df[["MAIN POSITION", "POSITION 2", "POSITION 3"]].values.ravel())
    posiciones = [p for p in posiciones if pd.notna(p)]

    for pos in posiciones:
        nombre = "POS_" + pos.upper().replace(" ", "_")
        df[nombre] = 0.0
        df.loc[df["POSITION 3"] == pos, nombre] = peso_pos3
        df.loc[df["POSITION 2"] == pos, nombre] = peso_pos2
        df.loc[df["MAIN POSITION"] == pos, nombre] = peso_main

    print(f"  Posiciones ponderadas: {len(posiciones)} columnas (main={peso_main}, pos2={peso_pos2}, pos3={peso_pos3})")
    return df

Ahora hacemos one- de la variable FOOT

In [19]:
def one_hot_foot(df):
    df = df.copy()
    for valor in ["Derecho", "Izquierdo", "Ambidiestro"]:
        df["FOOT_" + valor.upper()] = (df["FOOT"] == valor).astype(int)
    print(f"  One-hot de FOOT: 3 columnas")
    return df

Igual con Liga:

In [20]:
def one_hot_liga(df, col_id="LEAGUE_ID"):
    df = df.copy()

    ligas = sorted(df[col_id].dropna().unique())
    for liga in ligas:
        df[f"LEAGUE_{liga}"] = (df[col_id] == liga).astype(int)

    print(f"  One-hot de liga: {len(ligas)} columnas (a partir de {col_id})")
    return df

Ahora tratamos de asignaro peak age y young prospect

In [21]:
def crear_peak_age(df):
    df = df.copy()
    df["PEAK_AGE"] = df["AGE"].between(24, 28).astype(int)
    print(f"  PEAK_AGE creado ({df['PEAK_AGE'].sum()} jugadores en pico)")
    return df


def crear_young_prospect(df):
    df = df.copy()
    df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
    df["YOUNG_PROSPECT"] = (df["AGE"] <= 23).astype(int)
    print(f"  YOUNG_PROSPECT creado ({df['YOUNG_PROSPECT'].sum()} jóvenes promesas)")
    return df

### Normalizacion de las ligas y market value:

In [22]:
COLS_RENDIMIENTO = [
    # --- Ofensivo / finalización ---
    "GOALS_90",
    "ASSISTS_90",
    "SHOTS PER MATCH",
    "GOAL_CONVERSION",
    "xG/90",
    "xGDif",

    # --- Creación ---
    "KEY PASSES",
    "CROSS",
    "DRIBBLE_OF",
    "TACKLE RECEIVED",
    "SPACE BALL",          # pases al hueco

    # --- Pase / posesión ---
    "SUCCESSFUL PASSES (%)",
    "LONG PASS",
    "PromeP",              # pases promedio por partido

    # --- Defensivo ---
    "TACKLE",
    "INTERCEPTION",
    "CLEARANCE",
    "BLOCKS",
    "DRIBBLE_DEF",
    "INTENTIONAL OFFSIDES",  # fueras de juego provocados
]

In [23]:
def normalizar_por_liga(df, columnas):
    df = df.copy()
    for col in columnas:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[f"{col}_Z_LIGA"] = df.groupby("LEAGUE")[col].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0)
        )
    print(f"  Normalizadas por liga (z-score): {len(columnas)} columnas")
    return df

Capamos desviaciones muy anormales:

In [24]:
def capar_zscores(df, limite=3.0):
    df = df.copy()
    cols_z = [c for c in df.columns if c.endswith("_Z_LIGA")]
    df[cols_z] = df[cols_z].clip(lower=-limite, upper=limite)
    print(f"  Z-scores capados a ±{limite}: {len(cols_z)} columnas")
    return df

Ahora el market value:

In [25]:
def crear_log_market_value(df):
    df = df.copy()
    df["MARKET VALUE"] = pd.to_numeric(df["MARKET VALUE"], errors="coerce")
    df["LOG_MARKET_VALUE"] = np.log1p(df["MARKET VALUE"])
    print(f"  LOG_MARKET_VALUE creado (skew {df['LOG_MARKET_VALUE'].skew():.2f})")
    return df

Lo primero tenemos que agrupar por posiciones para poder dar scores en funcion de sus posiciones para ser capaces de encontrar a un 'jugador mejor' en los datos que realmente nos importan:

In [26]:
ROLES = {
    "Defensa central":      ["Defensa central"],
    "Lateral":              ["Lateral derecho", "Lateral izquierdo"],
    "Mediocentro defensivo":["Pivote", "Mediocentro"],
    "Media punta":          ["Mediocentro ofensivo", "Mediapunta",
                             "Interior derecho", "Interior izquierdo"],
    "Extremo":              ["Extremo derecho", "Extremo izquierdo"],
    "Delantero":            ["Delantero centro"],
}

Ahora a cada rol le damos las estadisticas mas importantes:


In [27]:
METRICAS_ROL = {
    "Defensa central": {
        "INTERCEPTION_Z_LIGA": 2, "CLEARANCE_Z_LIGA": 1.5, "TACKLE_Z_LIGA": 1.5,
        "BLOCKS_Z_LIGA": 1.5, "INTENTIONAL OFFSIDES_Z_LIGA": 1,
        "SUCCESSFUL PASSES (%)_Z_LIGA": 1, "LONG PASS_Z_LIGA": 0.5,
    },
    "Lateral": {
        "CROSS_Z_LIGA": 1.5, "TACKLE_Z_LIGA": 1.5, "INTERCEPTION_Z_LIGA": 1.5,
        "KEY PASSES_Z_LIGA": 1, "DRIBBLE_OF_Z_LIGA": 1, "ASSISTS_90_Z_LIGA": 1,
        "DRIBBLE_DEF_Z_LIGA": 1,
    },
    "Mediocentro defensivo": {
        "TACKLE_Z_LIGA": 2, "INTERCEPTION_Z_LIGA": 2, "SUCCESSFUL PASSES (%)_Z_LIGA": 1.5,
        "PromeP_Z_LIGA": 1, "LONG PASS_Z_LIGA": 1, "KEY PASSES_Z_LIGA": 0.5,
    },
    "Media punta": {
        "KEY PASSES_Z_LIGA": 2, "ASSISTS_90_Z_LIGA": 1.5, "SPACE BALL_Z_LIGA": 1.5,
        "DRIBBLE_OF_Z_LIGA": 1, "GOALS_90_Z_LIGA": 1, "xG/90_Z_LIGA": 1, "PromeP_Z_LIGA": 0.5,
    },
    "Extremo": {
        "DRIBBLE_OF_Z_LIGA": 2, "ASSISTS_90_Z_LIGA": 1.5, "KEY PASSES_Z_LIGA": 1.5,
        "GOALS_90_Z_LIGA": 1, "xG/90_Z_LIGA": 1, "CROSS_Z_LIGA": 1, "TACKLE RECEIVED_Z_LIGA": 0.5,
    },
    "Delantero": {
        "GOALS_90_Z_LIGA": 2, "xG/90_Z_LIGA": 1.5, "GOAL_CONVERSION_Z_LIGA": 1.5,
        "SHOTS PER MATCH_Z_LIGA": 1, "xGDif_Z_LIGA": 1, "ASSISTS_90_Z_LIGA": 0.5,
    },
}

Funciones para asignar estos roles:

In [28]:
def asignar_rol(df, roles=ROLES):
    df = df.copy()
    mapa = {}
    for rol, posiciones in roles.items():
        for pos in posiciones:
            mapa[pos] = rol
    df["ROL"] = df["MAIN POSITION"].map(mapa)
    print(f"  Roles asignados:")
    print(df["ROL"].value_counts().to_string().replace("\n", "\n    "))
    return df


def calcular_score_rol(df, metricas_rol=METRICAS_ROL):
    df = df.copy()
    df["SCORE_ROL"] = np.nan
    for rol, pesos_dict in metricas_rol.items():
        mask = df["ROL"] == rol
        cols = [c for c in pesos_dict if c in df.columns]
        pesos = np.array([pesos_dict[c] for c in cols])
        valores = df.loc[mask, cols].fillna(0).to_numpy()
        df.loc[mask, "SCORE_ROL"] = (valores * pesos).sum(axis=1) / pesos.sum()
    print(f"  SCORE_ROL calculado con pesos por rol")
    return df

Ponderamos el score por minutos:

In [29]:
def ponderar_score_por_minutos(df, min_referencia=2000):
    df = df.copy()
    # factor entre 0 y 1: satura en 1 al llegar a min_referencia minutos
    factor = (df["MINS"] / min_referencia).clip(upper=1.0)
    df["SCORE_ROL_AJUSTADO"] = df["SCORE_ROL"] * factor
    print(f"  SCORE_ROL_AJUSTADO calculado (referencia {min_referencia} min)")
    return df

PIPELINE: 

In [30]:
def pipeline_rellenar(config):

    # ============================================================
    # 1. CARGA DE DATOS
    # ============================================================
    print("Carga de datos")
    print("-" * 40)
    datos = cargar_datos(config["archivo_datos"], config["nombre_hoja"])
    
    # ============================================================
    # 2. RELLENO DE POSICIONES
    # ============================================================
    print("\nRelleno de posiciones")
    print("-" * 40)
    datos = rellenar_position_2(datos)
    datos = rellenar_position_3(datos)

    # ============================================================
    # 3. RELLENO DE HEIGHT
    # ============================================================
    print("\nRelleno de height")
    print("-" * 40)
    datos = rellenar_height_mediana(datos)

    # ============================================================
    # 4. RELLENO DE VALORES FIJOS
    # ============================================================
    print("\nRelleno de valores fijos")
    print("-" * 40)
    datos = rellenar_foot(datos)
    datos = rellenar_contract_until(datos)
    datos = rellenar_team_joined(datos)
    datos = rellenar_previous_team(datos)

        # ============================================================
    # 5. FILTRADO POR MINUTOS
    # ============================================================
    print("\nFiltrado por minutos jugados")
    print("-" * 40)
    datos = filtrar_mins(datos, minimo=900)

    # ============================================================
    # 5b. EXCLUSIÓN DE PORTEROS Y VALORES NULOS
    # ============================================================
    print("\nExclusión de porteros y valores de mercado nulos")
    print("-" * 40)
    datos = excluir_porteros(datos)
    datos = excluir_market_value_cero(datos)

        # ============================================================
    # 6. DIFERENCIA DE VALOR DE MERCADO
    # ============================================================
    print("\nCálculo de diferencia de valor de mercado")
    print("-" * 40)
    datos = calcular_diferencia_market_value(datos)
    
        # ============================================================
    # 7. PORCENTAJE DE TITULARIDADES
    # ============================================================
    print("\nCálculo de porcentaje de titularidades")
    print("-" * 40)
    datos = calcular_start_pct(datos)
    
        # ============================================================
    # 8. PORCENTAJE DE MINUTOS DISPONIBLES JUGADOS
    # ============================================================
    print("\nCálculo de porcentaje de minutos disponibles jugados")
    print("-" * 40)
    datos = calcular_available_min_pct(datos)
    
    # ============================================================
    # 9. CONTRIBUCIONES POR 90
    # ============================================================
    print("\nCálculo de contribuciones por 90 minutos")
    print("-" * 40)
    datos = calcular_contribuciones_90(datos)

    # ============================================================
    # 10. CONVERSIÓN DE GOL
    # ============================================================
    print("\nCálculo de conversión de gol")
    print("-" * 40)
    datos = calcular_goal_conversion(datos)

    # ============================================================
    # 11. SITUACIÓN CONTRACTUAL
    # ============================================================
    print("\nCálculo de situación contractual")
    print("-" * 40)
    datos = calcular_situacion_contractual(datos, fecha_referencia="30/06/2026")

    # ============================================================
    # 12. ÍNDICE DE ACCIONES DEFENSIVAS
    # ============================================================
    print("\nCálculo de acciones defensivas")
    print("-" * 40)
    datos = calcular_defensive_actions(datos)

    # ============================================================
    # 13. ÍNDICE DE CREACIÓN DE OCASIONES
    # ============================================================
    print("\nCálculo de creación de ocasiones")
    print("-" * 40)
    datos = calcular_chance_creation(datos)
    
    # ============================================================
    # 14. ÍNDICE DE AMENAZA CON BALÓN
    # ============================================================
    print("\nCálculo de amenaza con balón (regate + faltas recibidas)")
    print("-" * 40)
    datos = calcular_dribbling_threat(datos)
    
    # ============================================================
    # 16. CODIFICACIÓN DE CATEGÓRICAS (posición,pie, liga, edad)
    # ============================================================
    print("\nCodificación de posición ponderada y pie")
    print("-" * 40)
    datos = codificar_posiciones_ponderadas(datos)
    datos = one_hot_foot(datos)
    print("\nCodificación one-hot de liga")
    print("-" * 40)
    datos = one_hot_liga(datos, col_id="LEAGUE_ID")
    datos = crear_peak_age(datos)
    datos = crear_young_prospect(datos)

    # ============================================================
    # NORMALIZACIÓN POR LIGA (motor de la similitud)
    # ============================================================
    print("\nNormalización de métricas por liga (z-score)")
    print("-" * 40)
    datos = normalizar_por_liga(datos, COLS_RENDIMIENTO)
    datos = capar_zscores(datos, limite=7.0) 
    
    # ============================================================
    # NORMALIZACIÓN MARKET VALUE (motor de la similitud)
    # ============================================================
    
    datos = crear_log_market_value(datos)

    # ============================================================
    # ELIMINACIÓN DE COLUMNAS NO USADAS
    # ============================================================
    print("\nEliminación de columnas no usadas")
    print("-" * 40)
    datos = eliminar_columnas(datos, ["Despo", "RATING", "NATIONALITY", "PREVIOUS TEAM"])
        # ============================================================
    # SCORE DE RENDIMIENTO POR ROL
    # ============================================================
    print("\nAsignación de rol y score de rendimiento")
    print("-" * 40)
    datos = asignar_rol(datos)
    datos = calcular_score_rol(datos)
    datos = ponderar_score_por_minutos(datos, min_referencia=2000)
    
    print("\nPipeline completado")
    return datos


In [31]:
datos = pipeline_rellenar(config)

Carga de datos
----------------------------------------


  Datos cargados: 3406 filas x 48 columnas

Relleno de posiciones
----------------------------------------
  POSITION 2 rellenados con MAIN POSITION: 911
  POSITION 3 rellenados con POSITION 2: 1885

Relleno de height
----------------------------------------
  HEIGHT rellenados con mediana (1.84): 13

Relleno de valores fijos
----------------------------------------
  FOOT rellenados con 'Derecho': 16
  CONTRACT UNTIL rellenados con '30/06/2026': 128
  TEAM JOINED rellenados con '30/06/2025': 33
  PREVIOUS TEAM rellenados con 'unknown': 34

Filtrado por minutos jugados
----------------------------------------
  Filas eliminadas (MINS < 900): 1215
  Filas restantes: 2191

Exclusión de porteros y valores de mercado nulos
----------------------------------------
  Porteros eliminados: 161
  Jugadores de campo restantes: 2030
  Jugadores con valor 0 o nulo eliminados: 2
  Jugadores restantes: 2028

Cálculo de diferencia de valor de mercado
----------------------------------------
  MARKET_

In [32]:
#vemos cuantos valores vacios hay por columna
print("\nValores vacíos por columna:")
print(datos.isna().sum())



Valores vacíos por columna:
PLAYER_ID                      0
TEAM_ID                        0
LEAGUE_ID                      0
NAME                           0
TEAM                           0
                              ..
INTENTIONAL OFFSIDES_Z_LIGA    0
LOG_MARKET_VALUE               0
ROL                            0
SCORE_ROL                      0
SCORE_ROL_AJUSTADO             0
Length: 106, dtype: int64


In [33]:
datos.to_excel("datos_modelizacion.xlsx", index=False)

# MODELADO

In [34]:
# ============================================================
# 1. PREPARAR LA MATRIZ DE SIMILITUD
#    (se ejecuta UNA vez tras el preprocesado)
# ============================================================
def preparar_matriz_similitud(df, peso_rendimiento=0.5, peso_posicion=0.5):
    df = df.copy().reset_index(drop=True)

    cols_z   = [c for c in df.columns if c.endswith("_Z_LIGA")]
    cols_pos = [c for c in df.columns if c.startswith("POS_")]

    def estandarizar(M):
        return (M - M.mean(axis=0)) / (M.std(axis=0) + 1e-9)

    X_rend = estandarizar(df[cols_z].fillna(0).to_numpy())   * (peso_rendimiento ** 0.5)
    X_pos  = estandarizar(df[cols_pos].fillna(0).to_numpy()) * (peso_posicion ** 0.5)

    X = np.hstack([X_rend, X_pos])
    print(f"  Matriz lista: {X.shape[0]} jugadores x {X.shape[1]} dims "
          f"(rend={len(cols_z)} pos={len(cols_pos)}, pesos {peso_rendimiento}/{peso_posicion})")
    return df, X


# ============================================================
# 2. RECOMENDAR JUGADORES SIMILARES (con filtros duros)
# ============================================================
def recomendar(df, X, nombre_jugador, n=10,
               presupuesto_max=None,
               edad_min=None, edad_max=None,
               ligas_incluir=None, ligas_excluir=None,
               solo_contrato_acabando=False,
               misma_posicion=False):

    idx = df.index[df["NAME"] == nombre_jugador]
    if len(idx) == 0:
        print(f"  No encontrado: {nombre_jugador}")
        return None
    idx = idx[0]

    distancias = np.sqrt(((X - X[idx]) ** 2).sum(axis=1))

    res = df.copy()
    res["DISTANCIA"] = distancias
    res = res[res.index != idx]

    # ---- FILTROS DUROS ----
    if presupuesto_max is not None:
        res = res[res["MARKET VALUE"] <= presupuesto_max]
    if edad_min is not None:
        res = res[res["AGE"] >= edad_min]
    if edad_max is not None:
        res = res[res["AGE"] <= edad_max]
    if ligas_incluir is not None:
        res = res[res["LEAGUE"].isin(ligas_incluir)]
    if ligas_excluir is not None:
        res = res[~res["LEAGUE"].isin(ligas_excluir)]
    if solo_contrato_acabando:
        res = res[res["CONTRACT_OPPORTUNITY"] == 1]
    if misma_posicion:
        res = res[res["MAIN POSITION"] == df.loc[idx, "MAIN POSITION"]]

    if res.empty:
        print("  Ningún jugador cumple los filtros. Prueba a relajarlos.")
        return None

    cols = ["NAME", "TEAM", "LEAGUE", "MAIN POSITION", "AGE",
            "MARKET VALUE", "CONTRACT_OPPORTUNITY", "DISTANCIA"]
    return res.nsmallest(n, "DISTANCIA")[cols].reset_index(drop=True)


In [35]:
def mejorar_jugador(df, X, nombre_jugador, n=10,
                    n_similares=60,
                    max_ratio_valor=3.0,
                    margen_mejora_max=None,
                    presupuesto_max=None,
                    solo_contrato_acabando=False):

    # Localizar al jugador de referencia
    idx = df.index[df["NAME"] == nombre_jugador]
    if len(idx) == 0:
        print(f"  No encontrado: {nombre_jugador}")
        return None
    idx = idx[0]

    score_ref = df.loc[idx, "SCORE_ROL_AJUSTADO"]
    valor_ref = df.loc[idx, "MARKET VALUE"]
    rol_ref   = df.loc[idx, "ROL"]

    # 1. Partir de los N más SIMILARES (ancla de perfil realista)
    distancias = np.sqrt(((X - X[idx]) ** 2).sum(axis=1))
    res = df.copy()
    res["DISTANCIA"] = distancias
    res = res[res.index != idx]
    res = res.nsmallest(n_similares, "DISTANCIA")

    # 2. Mismo rol (mejorar tiene sentido dentro del puesto)
    res = res[res["ROL"] == rol_ref]

    # 3. Quedarse solo con los que MEJORAN el score
    res = res[res["SCORE_ROL_AJUSTADO"] > score_ref]
    res["MEJORA_SCORE"] = res["SCORE_ROL_AJUSTADO"] - score_ref

    # 4. ACOTAR el salto para que sea realista
    #    a) tope de valor de mercado relativo al jugador
    if max_ratio_valor is not None and valor_ref > 0:
        res = res[res["MARKET VALUE"] <= valor_ref * max_ratio_valor]
    #    b) tope absoluto de presupuesto (opcional)
    if presupuesto_max is not None:
        res = res[res["MARKET VALUE"] <= presupuesto_max]
    #    c) tope de cuánto mejor (evitar el crack a años luz)
    if margen_mejora_max is not None:
        res = res[res["MEJORA_SCORE"] <= margen_mejora_max]
    #    d) contrato acabándose (opcional)
    if solo_contrato_acabando:
        res = res[res["CONTRACT_OPPORTUNITY"] == 1]

    if res.empty:
        print("  Ningún jugador mejora al tuyo dentro de los límites. Prueba a relajarlos.")
        return None

    # 5. Ordenar por mejora de score (el que más mejora primero)
    cols = ["NAME", "TEAM", "LEAGUE", "MAIN POSITION", "AGE", "MARKET VALUE",
            "SCORE_ROL_AJUSTADO", "MEJORA_SCORE", "DISTANCIA"]
    return res.nlargest(n, "MEJORA_SCORE")[cols].reset_index(drop=True)

In [36]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import unicodedata

def _normaliza(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))

def lanzar_interfaz(df, X):
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    busqueda = widgets.Text(
        description="Buscar:", placeholder="Parte del nombre (ej. Lamine)",
        layout=widgets.Layout(width="400px"), style={"description_width": "initial"})
    desplegable = widgets.Dropdown(
        description="Jugador:", options=[],
        layout=widgets.Layout(width="400px"), style={"description_width": "initial"})

    modo = widgets.ToggleButtons(
        options=["Jugadores similares", "Mejorar a mi jugador"],
        description="Modo:", style={"description_width": "initial"})

    n_slider = widgets.IntSlider(
        value=10, min=3, max=30, description="Nº resultados:",
        layout=widgets.Layout(width="400px"), style={"description_width": "initial"})
    presupuesto = widgets.FloatText(
        value=0, description="Presupuesto máx (M, 0=sin límite):",
        layout=widgets.Layout(width="400px"), style={"description_width": "initial"})
    edad_max = widgets.IntText(
        value=0, description="Edad máx (0=sin límite):",
        layout=widgets.Layout(width="400px"), style={"description_width": "initial"})
    ratio_valor = widgets.FloatSlider(
        value=3.0, min=1.0, max=6.0, step=0.5,
        description="Máx. veces el valor (solo mejora):",
        layout=widgets.Layout(width="400px"), style={"description_width": "initial"})
    contrato = widgets.Checkbox(value=False, description="Solo contrato acabándose")
    boton = widgets.Button(description="Buscar", button_style="primary")
    salida = widgets.Output()

    def actualizar_opciones(*args):
        texto = _normaliza(busqueda.value)
        if not texto:
            desplegable.options = []
            return
        coincidencias = [nombre for nombre in df["NAME"] if texto in _normaliza(nombre)]
        desplegable.options = sorted(coincidencias)[:50]

    busqueda.observe(actualizar_opciones, names="value")

    def al_pulsar(b):
        with salida:
            clear_output()
            if not desplegable.value:
                print("Selecciona un jugador del desplegable.")
                return

            pres = presupuesto.value if presupuesto.value > 0 else None
            edad = edad_max.value if edad_max.value > 0 else None

            if modo.value == "Jugadores similares":
                resultado = recomendar(
                    df, X, desplegable.value, n=n_slider.value,
                    presupuesto_max=pres, edad_max=edad,
                    solo_contrato_acabando=contrato.value,
                )
            else:  # Mejorar a mi jugador
                resultado = mejorar_jugador(
                    df, X, desplegable.value, n=n_slider.value,
                    presupuesto_max=pres,
                    max_ratio_valor=ratio_valor.value,
                    solo_contrato_acabando=contrato.value,
                )

            if resultado is not None:
                display(resultado)

    boton.on_click(al_pulsar)
    display(busqueda, desplegable, modo, n_slider, presupuesto,
            edad_max, ratio_valor, contrato, boton, salida)

In [37]:
# Una vez, después del preprocesado:
datos, X = preparar_matriz_similitud(datos, peso_rendimiento=0.5, peso_posicion=0.5)

# Lanzar la interfaz:
lanzar_interfaz(datos, X)


  Matriz lista: 2028 jugadores x 33 dims (rend=20 pos=13, pesos 0.5/0.5)


Text(value='', description='Buscar:', layout=Layout(width='400px'), placeholder='Parte del nombre (ej. Lamine)…

Dropdown(description='Jugador:', layout=Layout(width='400px'), options=(), style=DescriptionStyle(description_…

ToggleButtons(description='Modo:', options=('Jugadores similares', 'Mejorar a mi jugador'), style=ToggleButton…

IntSlider(value=10, description='Nº resultados:', layout=Layout(width='400px'), max=30, min=3, style=SliderSty…

...

IntText(value=0, description='Edad máx (0=sin límite):', layout=Layout(width='400px'), style=DescriptionStyle(…

FloatSlider(value=3.0, description='Máx. veces el valor (solo mejora):', layout=Layout(width='400px'), max=6.0…

Checkbox(value=False, description='Solo contrato acabándose')

Button(button_style='primary', description='Buscar', style=ButtonStyle())

Output()

DETECTOR DE GANGAS

In [38]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score


def construir_features_gangas(df):
    # Columnas que NUNCA entran (leakage + identificadores + oportunidad)
    excluir = {
        "MARKET VALUE", "HIGHEST MARKET VALUE", "LOG_MARKET_VALUE",
        "MARKET_VALUE_DROP", "MARKET_VALUE_TREND_PCT",
        "PLAYER_ID", "TEAM_ID", "LEAGUE_ID", "NAME", "TEAM", "LEAGUE",
        "MAIN POSITION", "POSITION 2", "POSITION 3", "NATIONALITY",
        "BIRTH DATE", "CONTRACT UNTIL", "TEAM JOINED", "PREVIOUS TEAM",
        "FOOT", "ROL", "SCORE_ROL", "SCORE_ROL_AJUSTADO",
        "CONTRACT_OPPORTUNITY", "MONTHS_TO_CONTRACT_END",
    }

    # Métricas crudas que ya están como _Z_LIGA (evitar duplicar)
    crudas_con_z = [c for c in df.columns if f"{c}_Z_LIGA" in df.columns]
    excluir.update(crudas_con_z)

    # Nos quedamos solo con columnas numéricas no excluidas
    features = [c for c in df.columns
                if c not in excluir and pd.api.types.is_numeric_dtype(df[c])]

    print(f"  Features seleccionadas: {len(features)}")
    return features


def entrenar_modelos_valor(df, test_size=0.2, semilla=42):
    features = construir_features_gangas(df)

    X = df[features].fillna(0)
    y = df["LOG_MARKET_VALUE"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=semilla)

    modelos = {
        "RandomForest": RandomForestRegressor(
            n_estimators=400, max_depth=None, min_samples_leaf=3,
            random_state=semilla, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=400, max_depth=3, learning_rate=0.05,
            random_state=semilla),
    }

    resultados = {}
    for nombre, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        pred_test = modelo.predict(X_test)

        # Métricas en escala log
        mae_log = mean_absolute_error(y_test, pred_test)
        r2 = r2_score(y_test, pred_test)

        # MAE en millones reales (deshaciendo el log)
        mae_millones = mean_absolute_error(
            np.expm1(y_test), np.expm1(pred_test))

        print(f"\n  {nombre}")
        print(f"    R²:              {r2:.3f}")
        print(f"    MAE (log):       {mae_log:.3f}")
        print(f"    MAE (millones):  {mae_millones:.2f} M")

        resultados[nombre] = {
            "modelo": modelo, "features": features,
            "r2": r2, "mae_log": mae_log, "mae_millones": mae_millones}

    return resultados


import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score


def evaluar_modelos_cv(df, k=5, semilla=42):
    features = construir_features_gangas(df)   # la misma función de antes

    X = df[features].fillna(0)
    y = df["LOG_MARKET_VALUE"]

    cv = KFold(n_splits=k, shuffle=True, random_state=semilla)

    modelos = {
        "RandomForest": RandomForestRegressor(
            n_estimators=400, min_samples_leaf=3, random_state=semilla, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=400, max_depth=3, learning_rate=0.05, random_state=semilla),
    }

    predicciones = {}
    for nombre, modelo in modelos.items():
        # Predicción out-of-fold: cada jugador predicho por un modelo que no lo vio
        pred_oof_log = cross_val_predict(modelo, X, y, cv=cv, n_jobs=-1)

        r2  = r2_score(y, pred_oof_log)
        mae_log = mean_absolute_error(y, pred_oof_log)
        mae_millones = mean_absolute_error(np.expm1(y), np.expm1(pred_oof_log))

        print(f"\n  {nombre}  (validación cruzada {k}-fold)")
        print(f"    R²:              {r2:.3f}")
        print(f"    MAE (log):       {mae_log:.3f}")
        print(f"    MAE (millones):  {mae_millones:.2f} M")

        predicciones[nombre] = {
            "pred_oof_log": pred_oof_log,
            "features": features,
            "r2": r2, "mae_millones": mae_millones}

    return predicciones


def detectar_gangas_cv(df, prediccion_cv, n=20, metrica="ratio",
                        valor_min=None, valor_max=None):
    out = df.copy()
    out["VALOR_PREDICHO"] = np.expm1(prediccion_cv["pred_oof_log"]).round(2)
    out["VALOR_REAL"]     = out["MARKET VALUE"]

    # Infravaloración absoluta (en millones) y relativa (ratio)
    out["INFRAVALORACION"]     = (out["VALOR_PREDICHO"] - out["VALOR_REAL"]).round(2)
    out["RATIO_INFRAVALOR"]    = (out["VALOR_PREDICHO"] / out["VALOR_REAL"]).round(2)

    # Filtros opcionales de rango de precio (para acotar a lo fichable)
    if valor_min is not None:
        out = out[out["VALOR_REAL"] >= valor_min]
    if valor_max is not None:
        out = out[out["VALOR_REAL"] <= valor_max]

    orden = "RATIO_INFRAVALOR" if metrica == "ratio" else "INFRAVALORACION"

    cols = ["NAME", "TEAM", "LEAGUE", "MAIN POSITION", "AGE",
            "VALOR_REAL", "VALOR_PREDICHO", "INFRAVALORACION", "RATIO_INFRAVALOR"]
    return out.nlargest(n, orden)[cols].reset_index(drop=True)

def analizar_valor(df, prediccion_cv, n=20,
                   direccion="infravalorados",
                   metrica="log_residuo",        # "absoluto" | "ratio" | "log_residuo"
                   valor_min=None, valor_max=None,
                   edad_max=None, solo_contrato_acabando=False):

    out = df.copy()
    pred_log = prediccion_cv["pred_oof_log"]
    real_log = df["LOG_MARKET_VALUE"].to_numpy()

    out["VALOR_PREDICHO"] = np.expm1(pred_log).round(2)
    out["VALOR_REAL"]     = out["MARKET VALUE"]

    out["DIFERENCIA"]  = (out["VALOR_PREDICHO"] - out["VALOR_REAL"]).round(2)
    out["RATIO"]       = (out["VALOR_PREDICHO"] / out["VALOR_REAL"]).round(2)
    # Residuo en escala log: no está inflado por los valores altos
    out["LOG_RESIDUO"] = (pred_log - real_log).round(3)

    if valor_min is not None:
        out = out[out["VALOR_REAL"] >= valor_min]
    if valor_max is not None:
        out = out[out["VALOR_REAL"] <= valor_max]
    if edad_max is not None:
        out = out[out["AGE"] <= edad_max]
    if solo_contrato_acabando:
        out = out[out["CONTRACT_OPPORTUNITY"] == 1]

    col = {"absoluto": "DIFERENCIA", "ratio": "RATIO",
           "log_residuo": "LOG_RESIDUO"}[metrica]

    if direccion == "infravalorados":
        resultado = out.nlargest(n, col)
    else:
        resultado = out.nsmallest(n, col)

    cols = ["NAME", "TEAM", "LEAGUE", "MAIN POSITION", "AGE",
            "VALOR_REAL", "VALOR_PREDICHO", "DIFERENCIA", "RATIO", "LOG_RESIDUO"]
    return resultado[cols].reset_index(drop=True)

In [39]:
# Evaluar y comparar ambos modelos con validación cruzada
predicciones = evaluar_modelos_cv(datos, k=5)


  Features seleccionadas: 63

  RandomForest  (validación cruzada 5-fold)
    R²:              0.712
    MAE (log):       0.458
    MAE (millones):  6.59 M

  GradientBoosting  (validación cruzada 5-fold)
    R²:              0.732
    MAE (log):       0.438
    MAE (millones):  6.11 M


# CASOS DE USO

In [40]:
gb = predicciones["GradientBoosting"]

### EQUIPO CON PRESUPUESTO:

In [41]:
# 1. Equipo GRANDE busca gangas (absoluto: mayor margen en millones)
analizar_valor(datos, gb, direccion="infravalorados", metrica="absoluto")

,NAME,TEAM,LEAGUE,MAIN POSITION,AGE,VALOR_REAL,VALOR_PREDICHO,DIFERENCIA,RATIO,LOG_RESIDUO
0,Rayan Cherki,Manchester City,Premier League,Mediocentro ofensivo,22,65.0,174.20,109.20,2.68,0.976
1,Max Alleyne,Manchester City,Premier League,Defensa central,20,8.0,77.08,69.08,9.64,2.160
2,Mason Greenwood,Olympique de Marsella,Ligue 1,Extremo derecho,24,55.0,100.37,45.37,1.82,0.593
3,Marcus Tavernier,AFC Bournemouth,Premier League,Mediocentro ofensivo,27,23.0,54.12,31.12,2.35,0.831
4,Elliot Anderson,Nottingham Forest,Premier League,Mediocentro,23,60.0,90.67,30.67,1.51,0.407
5,Lucas Beraldo,París Saint-Germain FC,Ligue 1,Defensa central,22,20.0,48.56,28.56,2.43,0.859
6,Curtis Jones,Liverpool FC,Premier League,Mediocentro,25,35.0,62.15,27.15,1.78,0.562
7,James Garner,Everton FC,Premier League,Pivote,25,35.0,61.17,26.17,1.75,0.546
8,Marc Guéhi,Manchester City,Premier League,Defensa central,25,65.0,90.37,25.37,1.39,0.325
9,Rodrigo Zalazar,SC Braga,Liga Portugal,Mediocentro ofensivo,26,22.0,46.55,24.55,2.12,0.726


### EQUIPO CON POCO PRESUPUESTO:

In [42]:
# 2. Equipo MODESTO busca gangas (ratio, acotado a su presupuesto)
analizar_valor(datos, gb, direccion="infravalorados", metrica="ratio",
               valor_max=15)

,NAME,TEAM,LEAGUE,MAIN POSITION,AGE,VALOR_REAL,VALOR_PREDICHO,DIFERENCIA,RATIO,LOG_RESIDUO
0,Jaden Heskey,Sheffield Wednesday,Championship,Mediocentro,20,0.200,4.24,4.04,21.20,1.474
1,Abraham Marcus,CF Estrela Amadora,Liga Portugal,Extremo derecho,26,0.800,13.11,12.31,16.39,2.059
2,Max Alleyne,Manchester City,Premier League,Defensa central,20,8.000,77.08,69.08,9.64,2.160
3,Enzo Ebosse,Torino FC,Serie A,Defensa central,27,1.500,14.39,12.89,9.59,1.818
4,Danny Imray,West Bromwich Albion,Championship,Lateral derecho,22,0.275,2.61,2.34,9.49,1.041
5,Javi Sánchez,FC Arouca,Liga Portugal,Defensa central,29,0.800,7.03,6.23,8.79,1.496
6,Ángel Pérez,Deportivo Alavés,La Liga,Extremo derecho,23,1.000,8.56,7.56,8.56,1.564
7,José Fontán,FC Arouca,Liga Portugal,Defensa central,26,1.500,12.52,11.02,8.35,1.688
8,Jarvis Thornton,Sheffield Wednesday,Championship,Pivote,20,0.300,2.41,2.11,8.03,0.964
9,Guilherme Neiva,Avs Futebol,Liga Portugal,Extremo izquierdo,25,0.200,1.60,1.40,8.00,0.773


### JUGADORES SOBREVALORADOS:

In [43]:
# Sobrevalorados con residuo log, acotado a precio medio (donde el análisis es fiable)
analizar_valor(datos, predicciones["GradientBoosting"],
               direccion="sobrevalorados", metrica="log_residuo",
               valor_min=5, valor_max=40)

,NAME,TEAM,LEAGUE,MAIN POSITION,AGE,VALOR_REAL,VALOR_PREDICHO,DIFERENCIA,RATIO,LOG_RESIDUO
0,Pepê,FC Oporto,Liga Portugal,Extremo derecho,29,20.0,1.60,-18.40,0.08,-2.090
1,Rodrigo Mora,FC Oporto,Liga Portugal,Mediocentro ofensivo,19,38.0,4.94,-33.06,0.13,-1.882
2,Marquinhos,París Saint-Germain FC,Ligue 1,Defensa central,32,30.0,4.19,-25.81,0.14,-1.787
3,Gustavo Sá,FC Famalicão,Liga Portugal,Mediocentro ofensivo,21,18.0,2.66,-15.34,0.15,-1.647
4,Martim Fernandes,FC Oporto,Liga Portugal,Lateral derecho,20,12.0,1.60,-10.40,0.13,-1.610
5,Dodi Lukébakio,SL Benfica,Liga Portugal,Extremo derecho,28,20.0,3.31,-16.69,0.17,-1.583
6,Nico González,Atlético de Madrid,La Liga,Extremo izquierdo,28,24.0,4.15,-19.85,0.17,-1.580
7,Georgiy Sudakov,SL Benfica,Liga Portugal,Mediocentro ofensivo,23,28.0,5.10,-22.90,0.18,-1.559
8,Alan Varela,FC Oporto,Liga Portugal,Pivote,24,32.0,6.41,-25.59,0.20,-1.493
9,Moise Kean,Fiorentina,Serie A,Delantero centro,26,40.0,8.53,-31.47,0.21,-1.459


In [44]:
# 4. Sobrevalorados en ratio (predice mucho menos de lo que cuesta)
analizar_valor(datos, gb, direccion="sobrevalorados", metrica="ratio")

,NAME,TEAM,LEAGUE,MAIN POSITION,AGE,VALOR_REAL,VALOR_PREDICHO,DIFERENCIA,RATIO,LOG_RESIDUO
0,João Aurélio,CD Nacional,Liga Portugal,Lateral derecho,37,0.05,-0.26,-0.31,-5.20,-0.354
1,Aderllan Santos,Avs Futebol,Liga Portugal,Defensa central,37,0.10,-0.27,-0.37,-2.70,-0.414
2,André Geraldes,Casa Pia AC,Liga Portugal,Lateral derecho,35,0.20,-0.28,-0.48,-1.40,-0.507
3,Ponck,Avs Futebol,Liga Portugal,Defensa central,35,0.20,-0.23,-0.43,-1.15,-0.444
4,Liam Cooper,Sheffield Wednesday,Championship,Defensa central,34,0.30,-0.16,-0.46,-0.53,-0.439
5,Adam Forshaw,Blackburn Rovers,Championship,Mediocentro,34,0.20,-0.03,-0.23,-0.15,-0.209
6,Seba Pérez,Casa Pia AC,Liga Portugal,Pivote,33,0.80,-0.04,-0.84,-0.05,-0.633
7,Ben Mee,Sheffield United,Championship,Defensa central,36,0.50,0.02,-0.48,0.04,-0.388
8,Pepê,FC Oporto,Liga Portugal,Extremo derecho,29,20.00,1.60,-18.40,0.08,-2.090
9,Trent Alexander-Arnold,Real Madrid CF,La Liga,Lateral derecho,27,65.00,8.58,-56.42,0.13,-1.930


# USAMOS LOS DATOS DEL MODELO

### Incorporamos las predicciones a los datos:


In [45]:
def incorporar_predicciones(df, prediccion_cv):
    df = df.copy()
    pred_log = prediccion_cv["pred_oof_log"]
    df["VALOR_PREDICHO"] = np.expm1(pred_log).round(2)
    df["LOG_RESIDUO"]    = (pred_log - df["LOG_MARKET_VALUE"].to_numpy()).round(3)
    df["DIFERENCIA_VALOR"] = (df["VALOR_PREDICHO"] - df["MARKET VALUE"]).round(2)
    df["RATIO_VALOR"]      = (df["VALOR_PREDICHO"] / df["MARKET VALUE"]).round(2)
    # Tendencia legible: sube / baja / estable según el residuo log
    df["TENDENCIA"] = np.select(
        [df["LOG_RESIDUO"] > 0.20, df["LOG_RESIDUO"] < -0.20],
        ["Infravalorado (posible subida)", "Sobrevalorado (posible bajada)"],
        default="En línea con su valor")
    print("  Predicciones incorporadas al dataframe")
    return df

In [46]:
# Se ejecuta una vez, con el modelo elegido (el mejor: GradientBoosting)
datos = incorporar_predicciones(datos, predicciones["GradientBoosting"])

  Predicciones incorporadas al dataframe


### AHORA CREAMOS FICHAS DE JUGADORES Y DE EQUIPO:

In [47]:
def ficha_jugador(df, nombre_jugador):
    idx = df.index[df["NAME"] == nombre_jugador]
    if len(idx) == 0:
        print(f"  No encontrado: {nombre_jugador}")
        return None
    j = df.loc[idx[0]]

    # Percentil del score dentro de su rol (cómo de bueno es para su puesto)
    mismos_rol = df[df["ROL"] == j["ROL"]]
    pct_score = (mismos_rol["SCORE_ROL_AJUSTADO"] < j["SCORE_ROL_AJUSTADO"]).mean() * 100

    ficha = {
        "Nombre":            j["NAME"],
        "Equipo":            j["TEAM"],
        "Liga":              j["LEAGUE"],
        "Posición":          j["MAIN POSITION"],
        "Rol":               j["ROL"],
        "Edad":              int(j["AGE"]),
        "Minutos":           int(j["MINS"]),
        "Valor real (M)":    j["MARKET VALUE"],
        "Valor predicho (M)":j["VALOR_PREDICHO"],
        "Diferencia (M)":    j["DIFERENCIA_VALOR"],
        "Ratio":             j["RATIO_VALOR"],
        "Situación":         j["TENDENCIA"],
        "Score de rol":      round(j["SCORE_ROL_AJUSTADO"], 3),
        "Percentil en su rol": f"{pct_score:.0f}%",
        "Contrato hasta":    j["CONTRACT UNTIL"],
        "Meses contrato":    int(j["MONTHS_TO_CONTRACT_END"]) if pd.notna(j["MONTHS_TO_CONTRACT_END"]) else None,
    }
    return pd.Series(ficha)

In [48]:
def analizar_equipo(df, nombre_equipo, ordenar_por="DIFERENCIA_VALOR"):
    plantilla = df[df["TEAM"] == nombre_equipo].copy()
    if plantilla.empty:
        print(f"  Equipo no encontrado: {nombre_equipo}")
        return None

    cols = ["NAME", "MAIN POSITION", "ROL", "AGE",
            "MARKET VALUE", "VALOR_PREDICHO", "DIFERENCIA_VALOR",
            "RATIO_VALOR", "TENDENCIA", "SCORE_ROL_AJUSTADO",
            "MONTHS_TO_CONTRACT_END"]

    resultado = plantilla[cols].sort_values(ordenar_por, ascending=False)

    # Resumen rápido del equipo
    print(f"  {nombre_equipo}: {len(plantilla)} jugadores")
    print(f"  Valor total real:     {plantilla['MARKET VALUE'].sum():.1f} M")
    print(f"  Valor total predicho: {plantilla['VALOR_PREDICHO'].sum():.1f} M")
    infra = (plantilla['TENDENCIA'].str.contains('Infra')).sum()
    sobre = (plantilla['TENDENCIA'].str.contains('Sobre')).sum()
    print(f"  Infravalorados: {infra}  |  Sobrevalorados: {sobre}")

    return resultado.reset_index(drop=True)

In [49]:
# Ficha de un jugador
ficha_jugador(datos, "Eduardo Camavinga")



Nombre                              Eduardo Camavinga
Equipo                                 Real Madrid CF
Liga                                          La Liga
Posición                                  Mediocentro
Rol                             Mediocentro defensivo
Edad                                               23
Minutos                                          1527
Valor real (M)                                   50.0
Valor predicho (M)                              13.21
Diferencia (M)                                 -36.79
Ratio                                            0.26
Situación              Sobrevalorado (posible bajada)
Score de rol                                    0.564
Percentil en su rol                               70%
Contrato hasta                             30/06/2029
Meses contrato                                     36
dtype: object

### GANGAS + PARECIDOS

In [50]:
def gangas_similares(df, X, nombre_jugador, n=10,
                     n_similares=80,
                     ratio_min=1.3,
                     presupuesto_max=None,
                     mismo_rol=True):

    idx = df.index[df["NAME"] == nombre_jugador]
    if len(idx) == 0:
        print(f"  No encontrado: {nombre_jugador}")
        return None
    idx = idx[0]

    # 1. Los más similares por estilo
    distancias = np.sqrt(((X - X[idx]) ** 2).sum(axis=1))
    res = df.copy()
    res["DISTANCIA"] = distancias
    res = res[res.index != idx].nsmallest(n_similares, "DISTANCIA")

    # 2. Mismo rol (opcional)
    if mismo_rol:
        res = res[res["ROL"] == df.loc[idx, "ROL"]]

    # 3. Solo los infravalorados según el modelo
    res = res[res["RATIO_VALOR"] >= ratio_min]

    # 4. Presupuesto (opcional)
    if presupuesto_max is not None:
        res = res[res["MARKET VALUE"] <= presupuesto_max]

    if res.empty:
        print("  Ninguna ganga parecida dentro de los límites. Prueba a relajarlos.")
        return None

    # Ordenar por infravaloración relativa
    cols = ["NAME", "TEAM", "LEAGUE", "MAIN POSITION", "AGE",
            "MARKET VALUE", "VALOR_PREDICHO", "RATIO_VALOR", "DISTANCIA"]
    return res.nlargest(n, "RATIO_VALOR")[cols].reset_index(drop=True)

# PRODUCTIVIZAMOS LOS MODELOS:

In [51]:
def lanzar_buscador(df, X):
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    busqueda = widgets.Text(
        description="Buscar:", placeholder="Parte del nombre (ej. Lamine)",
        layout=widgets.Layout(width="420px"), style={"description_width": "initial"})
    desplegable = widgets.Dropdown(
        description="Jugador:", options=[],
        layout=widgets.Layout(width="420px"), style={"description_width": "initial"})

    modo = widgets.ToggleButtons(
        options=["Similares", "Mejorar", "Gangas parecidas"],
        description="Modo:", style={"description_width": "initial"})

    n_slider = widgets.IntSlider(
        value=10, min=3, max=30, description="Nº resultados:",
        layout=widgets.Layout(width="420px"), style={"description_width": "initial"})
    presupuesto = widgets.FloatText(
        value=0, description="Presupuesto máx (M, 0=sin límite):",
        layout=widgets.Layout(width="420px"), style={"description_width": "initial"})
    edad_max = widgets.IntText(
        value=0, description="Edad máx (0=sin límite):",
        layout=widgets.Layout(width="420px"), style={"description_width": "initial"})
    ratio_valor = widgets.FloatSlider(
        value=3.0, min=1.0, max=6.0, step=0.5,
        description="Máx. veces el valor (Mejorar):",
        layout=widgets.Layout(width="420px"), style={"description_width": "initial"})
    contrato = widgets.Checkbox(value=False, description="Solo contrato acabándose")
    boton = widgets.Button(description="Buscar", button_style="primary")
    salida = widgets.Output()

    def actualizar_opciones(*args):
        texto = _normaliza(busqueda.value)
        desplegable.options = (
            sorted([n for n in df["NAME"] if texto in _normaliza(n)])[:50]
            if texto else [])
    busqueda.observe(actualizar_opciones, names="value")

    def al_pulsar(b):
        with salida:
            clear_output()
            if not desplegable.value:
                print("Selecciona un jugador del desplegable.")
                return
            pres = presupuesto.value if presupuesto.value > 0 else None
            edad = edad_max.value if edad_max.value > 0 else None
            jugador = desplegable.value

            if modo.value == "Similares":
                r = recomendar(df, X, jugador, n=n_slider.value,
                               presupuesto_max=pres, edad_max=edad,
                               solo_contrato_acabando=contrato.value)
            elif modo.value == "Mejorar":
                r = mejorar_jugador(df, X, jugador, n=n_slider.value,
                                    presupuesto_max=pres,
                                    max_ratio_valor=ratio_valor.value,
                                    solo_contrato_acabando=contrato.value)
            else:  # Gangas parecidas
                r = gangas_similares(df, X, jugador, n=n_slider.value,
                                     presupuesto_max=pres)

            if r is not None:
                display(r)

    boton.on_click(al_pulsar)
    display(busqueda, desplegable, modo, n_slider, presupuesto,
            edad_max, ratio_valor, contrato, boton, salida)

In [52]:
def lanzar_explorador_mercado(df, prediccion_cv):
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    direccion = widgets.ToggleButtons(
        options=[("Infravalorados", "infravalorados"),
                 ("Sobrevalorados", "sobrevalorados")],
        description="Buscar:", style={"description_width": "initial"})
    metrica = widgets.ToggleButtons(
        options=[("Absoluto (M)", "absoluto"), ("Ratio", "ratio"),
                 ("Residuo log", "log_residuo")],
        description="Métrica:", style={"description_width": "initial"})
    n_slider = widgets.IntSlider(value=20, min=5, max=50, description="Nº:",
        style={"description_width": "initial"}, layout=widgets.Layout(width="400px"))
    valor_min = widgets.FloatText(value=0, description="Valor mín (M):",
        style={"description_width": "initial"}, layout=widgets.Layout(width="400px"))
    valor_max = widgets.FloatText(value=0, description="Valor máx (M, 0=sin):",
        style={"description_width": "initial"}, layout=widgets.Layout(width="400px"))
    edad_max = widgets.IntText(value=0, description="Edad máx (0=sin):",
        style={"description_width": "initial"}, layout=widgets.Layout(width="400px"))
    boton = widgets.Button(description="Explorar", button_style="primary")
    salida = widgets.Output()

    def al_pulsar(b):
        with salida:
            clear_output()
            r = analizar_valor(
                df, prediccion_cv, n=n_slider.value,
                direccion=direccion.value, metrica=metrica.value,
                valor_min=(valor_min.value if valor_min.value > 0 else None),
                valor_max=(valor_max.value if valor_max.value > 0 else None),
                edad_max=(edad_max.value if edad_max.value > 0 else None))
            if r is not None:
                display(r)

    boton.on_click(al_pulsar)
    display(direccion, metrica, n_slider, valor_min, valor_max, edad_max, boton, salida)

In [53]:
lanzar_buscador(datos, X)                                    # Bloque 1
lanzar_explorador_mercado(datos, predicciones["GradientBoosting"])  # Bloque 3
# (la ficha y el equipo del Bloque 2 ya los tienes)

Text(value='', description='Buscar:', layout=Layout(width='420px'), placeholder='Parte del nombre (ej. Lamine)…

Dropdown(description='Jugador:', layout=Layout(width='420px'), options=(), style=DescriptionStyle(description_…

ToggleButtons(description='Modo:', options=('Similares', 'Mejorar', 'Gangas parecidas'), style=ToggleButtonsSt…

IntSlider(value=10, description='Nº resultados:', layout=Layout(width='420px'), max=30, min=3, style=SliderSty…

FloatText(value=0.0, description='Presupuesto máx (M, 0=sin límite):', layout=Layout(width='420px'), style=Des…

...

FloatText(value=0.0, description='Valor mín (M):', layout=Layout(width='400px'), style=DescriptionStyle(descri…

FloatText(value=0.0, description='Valor máx (M, 0=sin):', layout=Layout(width='400px'), style=DescriptionStyle…

IntText(value=0, description='Edad máx (0=sin):', layout=Layout(width='400px'), style=DescriptionStyle(descrip…

Button(button_style='primary', description='Explorar', style=ButtonStyle())

Output()